# Stage 6 — Reproduce the training run

**Goal:** Starting from the same split, settings, and software versions, rebuild the chosen model and check that it gives the same predictions.

**Why:** A good score is useful only when we can trace and reproduce how it was made. This stage saves a replay record and a snapshot of the Colab software environment.

This is a **same-environment replay**. Later we will put the training code and dependency lock in version control and run it automatically.

In [ ]:
# Cell 2 — Why: select the exact saved run and reject changed files.
from google.colab import drive
from pathlib import Path
import hashlib, json, pandas as pd

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
run_id = "20260926T120702352046Z"
run_dir = root / "runs" / run_id
record = json.loads((run_dir / "run.json").read_text())
model_file = run_dir / "model.joblib"
if record["run_id"] != run_id or hashlib.sha256(model_file.read_bytes()).hexdigest() != record["model_sha256"]:
    raise ValueError("Wrong run or changed model file")
split_dir = root / "splits" / record["split_id"]
split = json.loads((split_dir / "manifest.json").read_text())
if split["split_id"] != record["split_id"]:
    raise ValueError("Wrong split manifest")
print("Original run:", run_id, "Split:", record["split_id"])

In [ ]:
# Cell 3 — Why: check that this runtime matches the original run.
import platform, numpy as np, sklearn, joblib

current = {"python": platform.python_version(), "pandas": pd.__version__,
           "numpy": np.__version__, "scikit_learn": sklearn.__version__,
           "joblib": joblib.__version__}
for package, version in record["versions"].items():
    if current[package] != version:
        raise RuntimeError(f"{package} changed: original {version}, now {current[package]}")
print("Core software versions match:", current)

In [ ]:
# Cell 4 — Why: verify and read the exact split before training.
frames = {}
for name, info in split["files"].items():
    raw = (split_dir / f"{name}.jsonl").read_bytes()
    if hashlib.sha256(raw).hexdigest() != info["sha256"] or len(raw.splitlines()) != info["rows"]:
        raise ValueError(f"{name} changed or is incomplete")
    frames[name] = pd.read_json(split_dir / f"{name}.jsonl", lines=True)
train, valid, test = (frames[name] for name in ["train", "validation", "test"])
features = record["features"]
if features != ["complaint_type", "borough", "month", "hour", "weekday"]:
    raise ValueError("Unexpected feature recipe")
print("Verified rows:", {name: len(frame) for name, frame in frames.items()})

In [ ]:
# Cell 5 — Why: rebuild the same preparation and model recipe.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

text_steps = Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="MISSING")),
                       ("encode", OneHotEncoder(handle_unknown="ignore"))])
time_steps = Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                       ("encode", OneHotEncoder(handle_unknown="ignore"))])
prepare = ColumnTransformer([("text", text_steps, ["complaint_type", "borough"]),
                             ("time", time_steps, ["month", "hour", "weekday"])])
recipe = Pipeline([("prepare", prepare),
                   ("model", LogisticRegression(solver=record["solver"], max_iter=1000,
                                                random_state=record["random_state"]))])
print("Rebuilt the recorded model recipe")

In [ ]:
# Cell 6 — Why: repeat the validation check without changing the recipe.
from sklearn.base import clone
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def scores(actual, predicted):
    return {"accuracy": round(accuracy_score(actual, predicted), 3),
            "precision_late": round(precision_score(actual, predicted, zero_division=0), 3),
            "recall_late": round(recall_score(actual, predicted, zero_division=0), 3),
            "f1_late": round(f1_score(actual, predicted, zero_division=0), 3)}

replayed_candidate = clone(recipe).fit(train[features], train["label"])
replayed_validation = scores(valid["label"], replayed_candidate.predict(valid[features]))
if replayed_validation != record["candidate_validation"]:
    raise ValueError("Validation scores differ from the original run")
print("Validation scores reproduced:", replayed_validation)

In [ ]:
# Cell 7 — Why: compare the rebuilt model with our saved trusted model.
earlier = pd.concat([train, valid], ignore_index=True)
replayed_model = clone(recipe).fit(earlier[features], earlier["label"])
original_model = joblib.load(model_file)  # Only load our own hash-verified artifact.
old_prob = original_model.predict_proba(test[features])[:, 1]
new_prob = replayed_model.predict_proba(test[features])[:, 1]
max_difference = float(np.max(np.abs(old_prob - new_prob)))
if not np.allclose(old_prob, new_prob, rtol=0, atol=1e-10):
    raise ValueError("Predictions changed; do not release this replay")
print("Predictions reproduced; largest probability difference:", max_difference)

In [ ]:
# Cell 8 — Why: capture the full installed package list for this replay.
from datetime import datetime, timezone
import subprocess, sys

replay_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
replay_dir = root / "replays" / replay_id
replay_dir.mkdir(parents=True, exist_ok=False)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        check=True, capture_output=True, text=True).stdout
environment_file = replay_dir / "pip-freeze.txt"
environment_file.write_text(freeze, encoding="utf-8")
print("Environment captured:", environment_file)

In [ ]:
# Cell 9 — Why: save proof that this run reproduced the original.
replay_record = {
    "replay_id": replay_id, "original_run_id": run_id,
    "split_id": record["split_id"], "source_dataset_id": record["source_dataset_id"],
    "original_model_sha256": record["model_sha256"],
    "environment_sha256": hashlib.sha256(environment_file.read_bytes()).hexdigest(),
    "versions": current, "validation_scores": replayed_validation,
    "prediction_rows_compared": len(test),
    "maximum_probability_difference": max_difference,
    "probability_tolerance": 1e-10, "lesson_version": "stage6-v1"
}
(replay_dir / "replay.json").write_text(json.dumps(replay_record, indent=2) + "\n", encoding="utf-8")
print("Replay ID:", replay_id)
print("Original run reproduced:", run_id)

# Stage 6 takeaway and quiz

**What you learned:** Reproducibility has several parts: the exact data, the exact model recipe, the software environment, and a check of the resulting predictions. Here we verified all four within the same Colab environment. A package list records what was installed; later we will put code and a dependency lock in version control for automated runs.

**Easy quiz:**

1. Why do we stop if a data file's SHA-256 differs?
2. Why do we compare software versions before replaying?
3. What does a near-zero probability difference tell us?
4. Does matching these predictions prove the model will always perform equally well on future months? Why?

Send me **Core software versions match**, **Validation scores reproduced**, **Predictions reproduced**, the **Replay ID**, and your answers.